# Voting Ensemble Across 5 Classifier Families on Kaggle

Clones `approach/catboot`. Trains 26 per-letter raw-feature classifiers
for each of 5 different classifier families -- CatBoost, XGBoost,
LightGBM, Random Forest, Logistic Regression -- then combines all of
them into one soft-voting ensemble: for each candidate letter, average
every family's predicted probability, guess whichever letter has the
highest average. Still falls back to dictionary frequency-matching first
whenever any train.txt word actually matches the board -- the voting
ensemble only kicks in for the out-of-dictionary case.

Kaggle ships CatBoost, XGBoost, and LightGBM preinstalled already;
Random Forest and Logistic Regression are plain sklearn. No GPU needed.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Internet = On** (needed to `git clone`). You can comment out any
`--model` line below to skip a family -- `VotingRawAgent` auto-detects
whichever families actually got trained and just uses those.

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/catboot"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train each classifier family

Each line is independent -- same 60,000-synthetic-state recipe, just a
different classifier library consuming it. Run whichever subset you want;
more families trained = a more real ensemble, but even one still works
(voting agent falls back to using just what's available).

In [ ]:
!python src/train_raw_classifier.py --model catboost --n-states 60000

In [ ]:
!python src/train_raw_classifier.py --model xgboost --n-states 60000

In [ ]:
!python src/train_raw_classifier.py --model lightgbm --n-states 60000

In [ ]:
!python src/train_raw_classifier.py --model random_forest --n-states 60000

In [ ]:
!python src/train_raw_classifier.py --model logreg --n-states 60000

## Validate the voting ensemble

Same held-out-train.txt methodology as every other branch. Prints which
families it actually found and used. Compare against
`approach/catboot`'s single-family CatBoost result and against
`approach/candidate-ngram`'s plain dictionary+ngram baseline (39-40%).

In [ ]:
!python src/validate_voting_raw.py --full

## Generate submission.csv

In [ ]:
!python src/generate_submission_voting_raw.py

## Save outputs

In [ ]:
import shutil, glob
for d in glob.glob("src/*_raw_models"):
    shutil.copytree(d, f"/kaggle/working/{d.split('/')[-1]}")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved all trained model directories + submission.csv to /kaggle/working/")